# <font color="#418FDE" size="6.5" uppercase>**Probability Calibration**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Evaluate probabilistic predictions using calibration curves and proper scoring metrics. 
- Calibrate classifiers with sigmoid and isotonic methods using cross-validation. 
- Apply isotonic regression for monotonic calibration-style relationships. 


## **1. Calibration Metrics**

### **1.1. Probability Estimates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_01.jpg?v=1783969014" width="250">



>* Probabilities express uncertainty beyond predicted labels
>* Risk estimates support better real-world decisions

>* Calibration means probabilities match real frequencies
>* Reliable probabilities guide risk-based decisions

>* Check ranking and probability accuracy separately
>* Reliable probabilities guide risk-based decisions



In [ ]:
#@title Python Code - Probability Estimates

# Probability estimates express uncertainty beyond class labels.
# Calibration checks whether probabilities match observed frequencies.
# This example compares confident and calibrated forecasts.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible binary outcomes and probability estimates.
rng = np.random.default_rng(42)
true_probability = 0.70
sample_count = 200

# Simulate outcomes from the true event probability.
y_true = rng.binomial(1, true_probability, sample_count)
confident_probs = np.full(sample_count, 0.90)
calibrated_probs = np.full(sample_count, 0.70)

# Define compact proper scoring metric functions.
def brier_score(y_values, p_values):
    return np.mean((p_values - y_values) ** 2)

# Compute observed frequency and Brier scores.
observed_rate = np.mean(y_true)
confident_brier = brier_score(y_true, confident_probs)
calibrated_brier = brier_score(y_true, calibrated_probs)

# Print a short comparison for learners.
print(f"Observed positive rate: {observed_rate:.2f}")
print(f"Overconfident estimate: {confident_probs[0]:.2f}")
print(f"Calibrated estimate: {calibrated_probs[0]:.2f}")
print(f"Overconfident Brier score: {confident_brier:.3f}")
print(f"Calibrated Brier score: {calibrated_brier:.3f}")

# Plot predicted probabilities against observed frequency.
labels = ["Overconfident", "Calibrated"]
predicted = [confident_probs[0], calibrated_probs[0]]
observed = [observed_rate, observed_rate]

# Build one simple calibration-style comparison plot.
plt.figure(figsize=(6, 4))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
plt.scatter(predicted, observed, s=120, label="Forecast groups")

# Add labels and display the single plot.
plt.xlabel("Predicted probability")
plt.ylabel("Observed positive frequency")
plt.title("Probability Estimates Need Calibration")
plt.legend()
plt.show()



### **1.2. Calibration Curves**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_02.jpg?v=1783969017" width="250">



>* Compare predicted probabilities with actual outcomes
>* Check whether probabilities express trustworthy uncertainty

>* Compare curves against the ideal diagonal
>* Departures show overconfidence or underconfidence

>* Check bin sizes before trusting curve patterns
>* Use calibration with discrimination and scoring metrics



In [ ]:
#@title Python Code - Calibration Curves

# Calibration curves compare probabilities with outcomes.
# Small bins make probability reliability visible.
# This example uses only built-in tools.

import math
import random
import matplotlib.pyplot as plt

# Make the random example reproducible.
random.seed(7)

# Create synthetic predicted probabilities.
n_samples = 240
predicted = [random.betavariate(2.0, 2.0) for _ in range(n_samples)]

# Simulate outcomes from slightly miscalibrated probabilities.
true_probs = [0.10 + 0.80 * (p ** 1.35) for p in predicted]
outcomes = [1 if random.random() < q else 0 for q in true_probs]

# Validate matching list lengths before binning.
assert len(predicted) == len(outcomes) == n_samples

# Prepare equal-width probability bins.
n_bins = 8
bin_edges = [i / n_bins for i in range(n_bins + 1)]

# Store average predictions and observed frequencies.
bin_predicted = []
bin_observed = []
bin_counts = []

# Compute calibration points for each bin.
for left, right in zip(bin_edges[:-1], bin_edges[1:]):
    in_bin = [i for i, p in enumerate(predicted) if left <= p < right]
    if right == 1.0:
        in_bin = [i for i, p in enumerate(predicted) if left <= p <= right]

    if len(in_bin) > 0:
        mean_pred = sum(predicted[i] for i in in_bin) / len(in_bin)
        mean_obs = sum(outcomes[i] for i in in_bin) / len(in_bin)
        bin_predicted.append(mean_pred)
        bin_observed.append(mean_obs)
        bin_counts.append(len(in_bin))

# Compute two proper scoring metrics.
eps = 1e-12
brier = sum((p - y) ** 2 for p, y in zip(predicted, outcomes)) / n_samples

# Clip probabilities for stable logarithms.
clipped = [min(max(p, eps), 1 - eps) for p in predicted]
log_loss = -sum(y * math.log(p) + (1 - y) * math.log(1 - p) for p, y in zip(clipped, outcomes)) / n_samples

# Print a compact numerical summary.
print(f"Samples: {n_samples}, bins used: {len(bin_counts)}")
print(f"Brier score: {brier:.3f}")
print(f"Log loss: {log_loss:.3f}")
print("Curve near diagonal means better calibration.")

# Plot the calibration curve against perfection.
plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
plt.plot(bin_predicted, bin_observed, "o-", label="Model calibration")
plt.xlabel("Average predicted probability")
plt.ylabel("Observed event frequency")
plt.title("Calibration Curve from Binned Probabilities")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



### **1.3. Brier Log Loss**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_03.jpg?v=1783969019" width="250">



>* Proper scores check probability reliability
>* Brier score measures squared probability error

>* Log loss heavily penalizes confident wrong predictions
>* Rewards honest uncertainty and calibrated probabilities

>* Brier and log loss reveal different strengths
>* Use curves and scores together



In [ ]:
#@title Python Code - Brier Log Loss

# Compare probability scores for calibration.
# Brier score rewards small probability errors.
# Log loss punishes confident wrong predictions.

import numpy as np
import matplotlib.pyplot as plt

# Store true binary outcomes.
y_true = np.array([1, 0, 1, 0, 1, 0, 1, 0])

# Compare cautious and overconfident probabilities.
y_cautious = np.array([0.70, 0.35, 0.65, 0.40, 0.75, 0.30, 0.60, 0.45])
y_overconfident = np.array([0.95, 0.05, 0.90, 0.10, 0.02, 0.08, 0.88, 0.12])

# Validate matching one dimensional arrays.
assert y_true.shape == y_cautious.shape == y_overconfident.shape
assert y_true.ndim == y_cautious.ndim == y_overconfident.ndim

# Define a small Brier score function.
def brier_score(y, p):
    return np.mean((p - y) ** 2)

# Define a safe log loss function.
def log_loss_binary(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# Calculate both proper scoring metrics.
brier_cautious = brier_score(y_true, y_cautious)
log_cautious = log_loss_binary(y_true, y_cautious)

# Calculate metrics for overconfident predictions.
brier_over = brier_score(y_true, y_overconfident)
log_over = log_loss_binary(y_true, y_overconfident)

# Print a compact comparison table.
print("Model              Brier score    Log loss")
print(f"Cautious model     {brier_cautious:.3f}          {log_cautious:.3f}")
print(f"Overconfident      {brier_over:.3f}          {log_over:.3f}")
print("Lower values are better for both metrics.")

# Plot predicted probabilities against outcomes.
plt.figure(figsize=(7, 4))
plt.scatter(range(len(y_true)), y_true, label="Actual outcome", color="black")
plt.plot(y_cautious, marker="o", label="Cautious probabilities")
plt.plot(y_overconfident, marker="s", label="Overconfident probabilities")

# Add labels for interpretation.
plt.ylim(-0.05, 1.05)
plt.xlabel("Case number")
plt.ylabel("Probability or outcome")
plt.title("Brier Score and Log Loss Compare Probability Quality")

# Show one clear teaching plot.
plt.legend()
plt.grid(alpha=0.3)
plt.show()



## **2. Calibration Methods**

### **2.1. Cross Validated Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_01.jpg?v=1783969025" width="250">



>* Calibration makes confidence scores match real outcomes
>* Cross validation prevents overconfident calibration bias

>* Use out-of-fold predictions for calibration
>* Refit model, then calibrate future probabilities

>* Separate ranking ability from probability accuracy
>* Choose folds carefully to avoid leakage



### **2.2. Sigmoid Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_02.jpg?v=1783969021" width="250">



>* Converts raw scores into better probabilities
>* Uses an S-shaped post-processing adjustment

>* Preserves ranking while reshaping probability scores
>* Learns adjustments from held-out outcome data

>* Use cross-validation for honest sigmoid calibration
>* Prefer isotonic for irregular probability corrections



### **2.3. Isotonic Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_03.jpg?v=1783969023" width="250">



>* Maps higher scores to higher probabilities
>* Flexible stepwise alternative to sigmoid calibration

>* Flexible mapping captures uneven score patterns
>* Needs enough data to avoid overfitting

>* Learn calibration from held-out predictions
>* Produce probabilities that match real-world outcomes



## **3. Advanced Calibration**

### **3.1. Frozen Estimator**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_01.jpg?v=1783969008" width="250">



>* Keep the trained model unchanged
>* Calibrate scores into reliable probabilities

>* Use unseen data for honest calibration
>* Isotonic mapping preserves score order

>* Separate scoring from probability calibration
>* Use representative data and evaluate carefully



In [ ]:
#@title Python Code - Frozen Estimator

# Demonstrate frozen estimator calibration.
# Use only NumPy and Matplotlib.
# Keep outputs short and visual.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic calibration data.
rng = np.random.default_rng(7)
n_samples = 240
raw_scores = rng.uniform(0.02, 0.98, n_samples)

# Simulate biased frozen model probabilities.
true_probs = 0.10 + 0.80 * raw_scores
true_probs = np.clip(true_probs, 0.01, 0.99)
outcomes = rng.binomial(1, true_probs)

# Freeze means scores never change.
frozen_scores = raw_scores.copy()
assert frozen_scores.shape == outcomes.shape

# Sort scores for monotonic calibration.
order = np.argsort(frozen_scores)
x_sorted = frozen_scores[order]
y_sorted = outcomes[order].astype(float)

# Implement simple isotonic regression pooling.
levels = []
weights = []
starts = []
ends = []

# Pool adjacent violators algorithm.
for index, value in enumerate(y_sorted):
    levels.append(float(value))
    weights.append(1.0)
    starts.append(index)
    ends.append(index)

    while len(levels) >= 2 and levels[-2] > levels[-1]:
        total_weight = weights[-2] + weights[-1]
        pooled_level = (levels[-2] * weights[-2] + levels[-1] * weights[-1]) / total_weight
        levels[-2] = pooled_level
        weights[-2] = total_weight
        ends[-2] = ends[-1]

        levels.pop()
        weights.pop()
        starts.pop()
        ends.pop()

# Expand pooled levels back to observations.
calibrated_sorted = np.empty_like(y_sorted)
for level, start, end in zip(levels, starts, ends):
    calibrated_sorted[start:end + 1] = level

# Restore original observation order.
calibrated_probs = np.empty_like(calibrated_sorted)
calibrated_probs[order] = calibrated_sorted

# Define compact Brier score function.
def brier_score(probabilities, labels):
    errors = (probabilities - labels) ** 2
    return float(np.mean(errors))

# Compare frozen scores and calibrated probabilities.
raw_brier = brier_score(frozen_scores, outcomes)
cal_brier = brier_score(calibrated_probs, outcomes)
print(f"Frozen estimator unchanged: {np.allclose(frozen_scores, raw_scores)}")
print(f"Brier before calibration: {raw_brier:.3f}")
print(f"Brier after isotonic calibration: {cal_brier:.3f}")
print(f"Number of isotonic steps learned: {len(levels)}")

# Plot monotonic calibration mapping.
plt.figure(figsize=(7, 4))
plt.scatter(frozen_scores, outcomes, s=14, alpha=0.25, label="Observed outcomes")
plt.step(x_sorted, calibrated_sorted, where="post", linewidth=2, label="Isotonic calibration")
plt.plot([0, 1], [0, 1], "--", linewidth=1, label="Perfect calibration")
plt.xlabel("Frozen estimator score")
plt.ylabel("Calibrated probability")
plt.title("Frozen estimator with isotonic calibration layer")
plt.legend(loc="best")
plt.tight_layout()
plt.show()



### **3.2. Multiclass Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_02.jpg?v=1783969010" width="250">



>* Calibration checks probabilities across multiple classes
>* Reliable distributions support better cost-sensitive decisions

>* Calibrate each class one-versus-rest
>* Isotonic mapping preserves ranking while correcting confidence

>* Normalize class probabilities for coherent decisions
>* Validate isotonic calibration to avoid overfitting



In [ ]:
#@title Python Code - Multiclass Calibration

# Multiclass calibration adjusts class probability estimates.
# Isotonic regression preserves score ordering relationships.
# This example uses only NumPy and Matplotlib.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(42)

# Create synthetic multiclass raw model scores.
n_samples, n_classes = 240, 3
raw_scores = rng.normal(size=(n_samples, n_classes))

# Convert raw scores into uncalibrated probabilities.
exp_scores = np.exp(raw_scores)
uncalibrated = exp_scores / exp_scores.sum(axis=1, keepdims=True)

# Sample true labels from a distorted probability pattern.
true_probs = uncalibrated ** np.array([0.7, 1.4, 1.0])
true_probs = true_probs / true_probs.sum(axis=1, keepdims=True)

# Draw one observed class for each example.
labels = np.array([rng.choice(n_classes, p=row) for row in true_probs])

# Validate the synthetic multiclass probability table.
assert uncalibrated.shape == (n_samples, n_classes)
assert np.allclose(uncalibrated.sum(axis=1), 1.0)

# Define a small isotonic regression calibrator.
def isotonic_fit_predict(x_train, y_train, x_test):
    order = np.argsort(x_train)
    x_sorted = x_train[order]
    y_sorted = y_train[order].astype(float)

    levels = y_sorted.copy()
    weights = np.ones_like(levels)
    starts = list(range(len(levels)))

    i = 0
    while i < len(levels) - 1:
        if levels[i] <= levels[i + 1]:
            i += 1
            continue

        total = weights[i] + weights[i + 1]
        merged = (levels[i] * weights[i] + levels[i + 1] * weights[i + 1]) / total
        levels[i] = merged
        weights[i] = total

        levels = np.delete(levels, i + 1)
        weights = np.delete(weights, i + 1)
        del starts[i + 1]
        i = max(i - 1, 0)

    fitted = np.empty_like(y_sorted)
    ends = starts[1:] + [len(y_sorted)]

    for start, end, level in zip(starts, ends, levels):
        fitted[start:end] = level

    return np.interp(x_test, x_sorted, fitted)

# Split data into calibration and test portions.
cal_size = 120
cal_scores = uncalibrated[:cal_size]
test_scores = uncalibrated[cal_size:]

# Keep matching labels for calibration and testing.
cal_labels = labels[:cal_size]
test_labels = labels[cal_size:]

# Calibrate each class using one-versus-rest targets.
calibrated = np.zeros_like(test_scores)
for class_id in range(n_classes):
    binary_targets = (cal_labels == class_id).astype(float)
    calibrated[:, class_id] = isotonic_fit_predict(
        cal_scores[:, class_id], binary_targets, test_scores[:, class_id]
    )

# Normalize classwise calibrated values into distributions.
row_sums = calibrated.sum(axis=1, keepdims=True)
calibrated = np.divide(calibrated, np.maximum(row_sums, 1e-12))

# Define multiclass Brier score for probability quality.
def multiclass_brier(probabilities, observed_labels):
    one_hot = np.eye(n_classes)[observed_labels]
    return np.mean(np.sum((probabilities - one_hot) ** 2, axis=1))

# Compare uncalibrated and calibrated probability scores.
brier_raw = multiclass_brier(test_scores, test_labels)
brier_cal = multiclass_brier(calibrated, test_labels)

# Print a compact teaching summary.
print(f"Samples: {n_samples}, classes: {n_classes}")
print(f"Raw multiclass Brier score: {brier_raw:.3f}")
print(f"Calibrated Brier score: {brier_cal:.3f}")
print("Each class was calibrated one-versus-rest, then normalized.")

# Plot class zero calibration before and after.
bins = np.linspace(0, 1, 7)
raw_bin = np.digitize(test_scores[:, 0], bins) - 1
cal_bin = np.digitize(calibrated[:, 0], bins) - 1

# Compute observed frequencies inside probability bins.
def bin_curve(scores, bin_ids):
    centers, observed = [], []
    for bin_id in range(len(bins) - 1):
        mask = bin_ids == bin_id
        if np.any(mask):
            centers.append(scores[mask].mean())
            observed.append((test_labels[mask] == 0).mean())
    return np.array(centers), np.array(observed)

# Build curves for one displayed class.
raw_x, raw_y = bin_curve(test_scores[:, 0], raw_bin)
cal_x, cal_y = bin_curve(calibrated[:, 0], cal_bin)

# Draw exactly one calibration plot.
plt.figure(figsize=(6, 4))
plt.plot([0, 1], [0, 1], "k--", label="Perfect")
plt.plot(raw_x, raw_y, "o-", label="Raw class 0")
plt.plot(cal_x, cal_y, "s-", label="Calibrated class 0")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("One-versus-rest multiclass calibration")
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Isotonic Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_03.jpg?v=1783969012" width="250">



>* Maps higher scores to higher probabilities
>* Preserves ranking while improving probability calibration

>* Flexible steps capture irregular calibration patterns
>* Needs enough data to avoid noise

>* Calibrate using held-out or cross-validated predictions
>* Use representative data to adjust probabilities



In [ ]:
#@title Python Code - Isotonic Regression

# Demonstrate isotonic regression calibration.
# Use only NumPy and Matplotlib.
# Keep outputs short and visual.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic score and outcome data.
rng = np.random.default_rng(7)
scores = np.linspace(0.02, 0.98, 80)

# Simulate miscalibrated event probabilities.
true_probs = 0.10 + 0.80 * scores**1.7
outcomes = rng.binomial(1, true_probs)

# Sort observations by model score.
order = np.argsort(scores)
x_sorted = scores[order]
y_sorted = outcomes[order].astype(float)

# Implement pool adjacent violators algorithm.
levels = y_sorted.copy().tolist()
weights = np.ones_like(y_sorted).tolist()
starts = list(range(len(y_sorted)))

# Merge neighboring blocks that violate monotonicity.
i = 0
while i < len(levels) - 1:
    if levels[i] <= levels[i + 1]:
        i += 1
    else:
        total = weights[i] + weights[i + 1]
        avg = (levels[i] * weights[i] + levels[i + 1] * weights[i + 1]) / total
        levels[i:i + 2] = [avg]
        weights[i:i + 2] = [total]
        starts[i:i + 2] = [starts[i]]
        i = max(i - 1, 0)

# Expand block levels into calibrated probabilities.
calibrated = np.empty_like(y_sorted)
for block_index, level in enumerate(levels):
    start = starts[block_index]
    stop = starts[block_index + 1] if block_index + 1 < len(starts) else len(y_sorted)
    calibrated[start:stop] = level

# Validate monotonic calibrated probabilities.
is_monotonic = np.all(np.diff(calibrated) >= -1e-12)
brier_raw = np.mean((scores - outcomes) ** 2)

# Compute calibrated Brier score.
calibrated_original = np.empty_like(calibrated)
calibrated_original[order] = calibrated
brier_cal = np.mean((calibrated_original - outcomes) ** 2)

# Print a compact teaching summary.
print(f"Observations: {len(scores)}")
print(f"Monotonic calibration learned: {is_monotonic}")
print(f"Raw Brier score: {brier_raw:.3f}")
print(f"Isotonic Brier score: {brier_cal:.3f}")

# Plot raw scores and isotonic mapping.
plt.figure(figsize=(7, 4))
plt.scatter(scores, outcomes, alpha=0.35, label="Observed outcomes")
plt.plot(x_sorted, calibrated, color="crimson", linewidth=2.5, label="Isotonic mapping")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
plt.xlabel("Original model score")
plt.ylabel("Calibrated probability")
plt.title("Isotonic Regression Learns a Monotonic Calibration Curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Probability Calibration**</font>


In this lecture, you learned to:
- Evaluate probabilistic predictions using calibration curves and proper scoring metrics. 
- Calibrate classifiers with sigmoid and isotonic methods using cross-validation. 
- Apply isotonic regression for monotonic calibration-style relationships. 

In the next Module (Module 16), we will go over 'Decision Trees'